In [1]:
import pandas as pd
url = "https://raw.githubusercontent.com/ogut77/DataScience/master/insurance.csv"
df = pd.read_csv(url)


In [2]:
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


Context in Insurance Data
This dataset is often used to predict charges based on the other variables (age, sex, bmi, children, smoker, region). For example:

Input Variables (X): age, sex, bmi, children, smoker, region (features used to make predictions).

Output Variable (y): charges (what you’re trying to predict).

Describtion of variables
1. Age
Description: The age of the individual (the insured person).
Type: Numerical (integer).
Example Values: 19, 45, 62, etc.
Role in Insurance: Age is a key factor in determining insurance charges. Older individuals often have higher medical costs (and thus higher charges) due to increased health risks.
2. Sex
Description: The gender of the individual.
Type: Categorical (text or binary).
Example Values: "male," "female"
Role in Insurance: Gender can influence insurance charges because health risks and medical expenses may differ between males and females (e.g., pregnancy-related costs for females).
3. BMI (Body Mass Index)
Description: A measure of body fat based on height and weight (calculated as weight in kg divided by height in meters squared).
Type: Numerical (float).
Example Values: 25.3, 30.1, 18.5, etc.
Role in Insurance: Higher BMI often correlates with increased health risks (e.g., obesity-related conditions like diabetes or heart disease), leading to higher insurance charges.
4. Children
Description: The number of children (dependents) covered under the individual’s insurance plan.
Type: Numerical (integer).
Example Values: 0, 1, 3, etc.
Role in Insurance: More children can increase insurance costs slightly, as it may reflect additional healthcare needs, though the effect is often less pronounced than other factors like smoking or age.
5. Smoker
Description: Indicates whether the individual smokes tobacco.
Type: Categorical (text or binary).
Example Values: "yes," "no" .
Role in Insurance: Smoking is a major factor in insurance charges. Smokers typically have much higher medical costs due to risks like lung disease or cancer, so their charges are significantly elevated.
6. Region
Description: The geographic region where the individual lives.
Type: Categorical (text).
Example Values: "northeast," "southeast," "southwest," "northwest" (common in U.S.-based datasets).
Role in Insurance: Charges can vary by region due to differences in healthcare costs, lifestyle factors, or local insurance regulations.
7. Charges
Description: The insurance charges (or premiums/costs) billed to the individual, typically in a currency like USD.
Type: Numerical (float).
Example Values: 1684.52, 11234.89, 32050.23, etc.
Role in Insurance: This is usually the target variable (output) in predictive modeling. It represents the amount the insurance company charges, influenced by all the other columns (age, sex, BMI, etc.).



In [3]:
#1. Check if there is null value in dataset df (5 pt)
df.isnull().sum()

,0
age,0
sex,0
bmi,0
children,0
smoker,0
region,0
charges,0


In [12]:
#2. Assign charges to y  and others to X using df. y is output variable and X is input variables (5 pt)

# assigning output variable (y)
y = df['charges']

# assigning input variables (x)
X = df.drop('charges', axis=1)

#to check the results
print(X.head())
print(y.head())

   age     sex     bmi  children smoker     region
0   19  female  27.900         0    yes  southwest
1   18    male  33.770         1     no  southeast
2   28    male  33.000         3     no  southeast
3   33    male  22.705         0     no  northwest
4   32    male  28.880         0     no  northwest
0    16884.92400
1     1725.55230
2     4449.46200
3    21984.47061
4     3866.85520
Name: charges, dtype: float64


In [11]:
#3. Use  get_dummies() function from the pandas library to convert categorical variables in a DataFrame (X).
# Drop first drops the first category’s dummy variable to avoid multicollinearity (5 pt)

X = pd.get_dummies(X, drop_first=True)

#to check the result
X.head()

,age,bmi,children,sex_male,smoker_yes,region_northwest,region_southeast,region_southwest
0,19,27.900,0,False,True,False,False,True
1,18,33.770,1,True,False,False,True,False
2,28,33.000,3,True,False,False,True,False
3,33,22.705,0,True,False,True,False,False
4,32,28.880,0,True,False,True,False,False


In [13]:
#Use following methods for the evaluation on test and train data
def evalmetric(y,ypred):
 from scipy.stats import pearsonr
 import numpy as np
 e = y - ypred
 mse_f = np.mean(e**2)
 rmse_f = np.sqrt(mse_f)
 mae_f = np.mean(abs(e))
 mape_f = 100*np.mean(abs(e/y))
 crl, _ = pearsonr(y, ypred)
 r2_f = crl*crl
 print("MSE:", mse_f)
 print("RMSE:", rmse_f)
 print("MAE:",mae_f)
 print("MAPE:",mape_f)
 print("R-Squared:", round(r2_f, 4))


In [17]:
#4.Get the correlation between X variables and y variables.(5 pt)
# first to combine X and y temporarily
X = pd.get_dummies(X, drop_first=True)
# comombine X and y
df_corr = X.copy()
df_corr['charges'] = y

# correlation with charges
df_corr.corr()['charges']


,charges
age,0.299008
bmi,0.198341
children,0.067998
sex_male,0.057292
smoker_yes,0.787251
region_northwest,-0.039905
region_southeast,0.073982
region_southwest,-0.043210
charges,1.000000


In [20]:
#5.Split a dataset into 25%  of data as test data  and 75% of data as training data ( pt)

from sklearn.model_selection import train_test_split

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

#to verify

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(1003, 8)
(335, 8)
(1003,)
(335,)


In [24]:
#6. Using Decision Tree and Linear Regression methods, compare the performance results on both test and training data
#to determine which one is more likely to overfit and which is more likely to underfit.
# Do you think that Lasso and Ridge regularization are more likely to improve the results of Linear model test data,
# or would Random Forest or Boosting methods are more likely to improve the results of Decison tree test data?
#Explain your reasoning.(35 pt)
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score

#TRAINING linear reg:
# Linear regression model
lr = LinearRegression()
lr.fit(X_train, y_train)

# ppredictions
y_train_pred_lr = lr.predict(X_train)
y_test_pred_lr = lr.predict(X_test)

# R2 scores
lr_train_score = r2_score(y_train, y_train_pred_lr)
lr_test_score = r2_score(y_test, y_test_pred_lr)

print("Linear Regression Train R2:", lr_train_score)
print("Linear Regression Test R2:", lr_test_score)

#Training decision tree
# Decision Tree model
dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)

# ppredictions
y_train_pred_dt = dt.predict(X_train)
y_test_pred_dt = dt.predict(X_test)

# R2 scores
dt_train_score = r2_score(y_train, y_train_pred_dt)
dt_test_score = r2_score(y_test, y_test_pred_dt)

print("Decision Tree Train R2:", dt_train_score)
print("Decision Tree Test R2:", dt_test_score)

#EXPLANATION ANSWERR
#The reason why decision tree overfits is that it perfectly matches the training data but is poor in the test data. Linear Regression is more consistent but can be underfitting a little. Linear Regression can be enhanced with regularization techniques such as Ridge or Lasso that may lessen the variance of each coefficient. In the case of Decision Trees, ensemble algorithms like Random Forest or Boosting tend to enhance the performance of the tests through the minimization of the variance and the maximization of the model stability.

Linear Regression Train R2: 0.7449555328228536
Linear Regression Test R2: 0.7672642952734356
Decision Tree Train R2: 0.9987411422200098
Decision Tree Test R2: 0.7575720467376498


In [25]:
#7. Explain performance of linear regressin on test data
# using  Root mean squared error, mean absolute error, mean absolute percentage error and R2 metric (10 pt)
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# Predict test data using Linear Regression
y_pred = lr.predict(X_test)

# RMSE
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

# MAE
mae = mean_absolute_error(y_test, y_pred)

# MAPE
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

# R2
r2 = r2_score(y_test, y_pred)

print("RMSE:", rmse)
print("MAE:", mae)
print("MAPE:", mape)
print("R2 Score:", r2)

#The Linear Regression model accounts 76.7 percent of the variation in insurance charges as the model has a relatively good fit as indicated by the R2 of 0.767.
#The value of RMSE (5926) shows that the average deviation is approximately 5926 and due to more intensive penalization of big values,
#it implies that there are more severe deviations of certain predictions. According to the MAE (4243), on average,
#there is a difference of 4243 between predictions and the actual charges.
#The MAPE 44.47% value suggests that the model has average predictive accuracy with an average error of 44.47% i.e. the model is mediocre thus can be enhanced.


RMSE: 5926.023602394469
MAE: 4243.654116653137
MAPE: 44.468185116980976
R2 Score: 0.7672642952734356


In [34]:
#8. Use Random Forest and Boosting methods (XGBoost, LightGBM, and CatBoost)
#to obtain the evaluation scores on  test data.
#Which Boosting technique yielded the best performance on the test data based on the R² metric?
#Did you achieve a better result compared to Random Forest on the test data based on the R² metric?
#If there is improvement on Random forest or boosting methods over decison tree, explain  (30 pt)

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
#RF
rf = RandomForestRegressor(random_state=42)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_r2 = r2_score(y_test, rf_pred)

print("Random Forest R2:", rf_r2)

#XG
xgb = XGBRegressor(random_state=42)
xgb.fit(X_train, y_train)

xgb_pred = xgb.predict(X_test)
xgb_r2 = r2_score(y_test, xgb_pred)

print("XGBoost R2:", xgb_r2)
#LGBM
lgbm = LGBMRegressor(random_state=42)
lgbm.fit(X_train, y_train)

lgbm_pred = lgbm.predict(X_test)
lgbm_r2 = r2_score(y_test, lgbm_pred)

print("LightGBM R2:", lgbm_r2)
#CATBOOST
cat = CatBoostRegressor(verbose=0, random_state=42)
cat.fit(X_train, y_train)

cat_pred = cat.predict(X_test)
cat_r2 = r2_score(y_test, cat_pred)

print("CatBoost R2:", cat_r2)

#Among the boosting techniques, CatBoost achieved the best performance on the test data with an R² score of 0.8586, followed by LightGBM (0.8542) and XGBoost (0.8248). CatBoost slightly outperformed Random Forest (0.8468) on the test data.

#This indicates that boosting methods can provide better predictive performance than Random Forest and Decision Tree models, because boosting algorithms build trees sequentially and focus on correcting the errors made by previous models. As a result, they often produce more accurate and robust predictions compared to a single decision tree or even Random Forest in some cases.

Random Forest R2: 0.8468139999098687
XGBoost R2: 0.8248177912664716
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000200 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 319
[LightGBM] [Info] Number of data points in the train set: 1003, number of used features: 8
[LightGBM] [Info] Start training from score 13267.935814
LightGBM R2: 0.8541656057996099
CatBoost R2: 0.8585679825166035
